In [10]:
# save_final_model.py
# Run this once to train the final model and save it for the dashboard to use.

import pandas as pd
import joblib
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------------
# 1. Load the final model-ready dataset
# ------------------------------------------------------------------
DATA_PATH = "../data/model_ready_cart_sessions_final.csv"  # adjust if needed
MODEL_OUTPUT_PATH = "../models/final_model.pkl"
FEATURES_OUTPUT_PATH = "../models/feature_columns.pkl"

df = pd.read_csv(DATA_PATH)
print("Loaded shape:", df.shape)

# ------------------------------------------------------------------
# 2. Drop the low-impact features identified by SHAP analysis
# ------------------------------------------------------------------
low_impact_features = [
    "no_view_before_cart",
    "never_viewed_cart_item_before",
    "cart_to_view_ratio",
    "views_before_cart",
]
X = df.drop(columns=["reconstructed_session_id", "is_abandoned"] + low_impact_features)
y = df["is_abandoned"]

feature_columns = X.columns.tolist()
print("Number of features used by the model:", len(feature_columns))

# ------------------------------------------------------------------
# 3. Train the final model using the best hyperparameters found earlier
# ------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

final_model = GradientBoostingClassifier(
    n_estimators=392,
    learning_rate=0.0195,
    max_depth=4,
    min_samples_leaf=28,
    subsample=0.892,
    random_state=42,
)
final_model.fit(X_train, y_train)

print("Training accuracy:", final_model.score(X_train, y_train))
print("Test accuracy:", final_model.score(X_test, y_test))

# ------------------------------------------------------------------
# 4. Save the model and the exact feature column order for the dashboard
# ------------------------------------------------------------------
joblib.dump(final_model, MODEL_OUTPUT_PATH)
joblib.dump(feature_columns, FEATURES_OUTPUT_PATH)

print("\nSaved model to:", MODEL_OUTPUT_PATH)
print("Saved feature column list to:", FEATURES_OUTPUT_PATH)

Loaded shape: (42853, 27)
Number of features used by the model: 21
Training accuracy: 0.608161717519398
Test accuracy: 0.5917629214794073

Saved model to: ../models/final_model.pkl
Saved feature column list to: ../models/feature_columns.pkl
